# Homework: English-to-German Translation with Transformers

In this assignment, you will build a complete **encoder-decoder Transformer** from scratch and use it to translate English sentences into German.

By the end of this notebook you will have:
- Built vocabularies for both languages
- Implemented a PyTorch `Dataset` for parallel text data
- Coded sinusoidal positional encoding
- Assembled a full sequence-to-sequence Transformer model
- Written a training loop with teacher forcing
- Implemented autoregressive greedy decoding for translation

## Prerequisites

Make sure you have watched the videos.

## Tasks Overview

| Task | Topic |
|------|-------|
| Task 1 | Build Vocabularies |
| Task 2 | Dataset and DataLoader |
| Task 3 | Positional Encoding |
| Task 4 | Seq2Seq Transformer Model |
| Task 5 | Training Loop |
| Task 6 | Greedy Translation |

## Setup

Run the cell below to import all required libraries and define the constants used throughout the notebook.

In [ ]:
from datasets import load_dataset
import torch
import torch.nn as nn
import torch.optim as optim
import math
import random
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Model hyperparameters
D_MODEL = 256          # Embedding / hidden dimension
NHEAD = 4              # Number of attention heads
NUM_LAYERS = 2         # Number of encoder and decoder layers
DIM_FEEDFORWARD = 512  # Feed-forward hidden size
DROPOUT = 0.1
MAX_SEQ_LEN = 30       # Maximum sequence length (tokens)

# Training hyperparameters
BATCH_SIZE = 64
NUM_EPOCHS = 15
LEARNING_RATE = 0.0003

# Number of training pairs to use (increase for better translations, up to ~29,000)
N_TRAIN = 5000

# Special token indices (reserved positions in every vocabulary)
PAD_IDX = 0  # padding — used to fill short sequences in a batch
UNK_IDX = 1  # unknown — used for words not seen during training
SOS_IDX = 2  # start-of-sequence — prepended to every target sentence
EOS_IDX = 3  # end-of-sequence — appended to every sentence

Using device: cpu


## Data

We use the **Multi30k** dataset — a standard benchmark for neural machine translation containing ~29,000 English–German sentence pairs describing everyday images. Sentences are short (average ~12 words) and cover varied, real-world vocabulary.

Run the cell below to download and load the data. It will be cached locally after the first download.

> **Training time note:** With `N_TRAIN = 5000` pairs, training takes approximately 5–15 minutes on a CPU. On a GPU or Google Colab it finishes in under a minute. You can increase `N_TRAIN` up to ~29,000 for better translations if you have more time.

In [ ]:
raw = load_dataset("bentrevett/multi30k")

# Build list of (English, German) pairs from the training split
parallel_data = [(ex["en"], ex["de"]) for ex in raw["train"]]
parallel_data = parallel_data[:N_TRAIN]
random.shuffle(parallel_data)

print(f"Total training pairs: {len(parallel_data)}")
print("\nSample pairs:")
for en, de in random.sample(parallel_data, 5):
    print(f"  EN: {en}")
    print(f"  DE: {de}")
    print()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/4.60M [00:00<?, ?B/s]

val.jsonl:   0%|          | 0.00/164k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/156k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Total training pairs: 5000

Sample pairs:
  EN: A woman is talking on her cellphone while a couple id sitting at a restaurant.
  DE: Eine Frau spricht am Handy, während ein Paar im Restaurant sitzt.

  EN: A woman with short blond-hair rises from a chair as another woman in a burgundy shirt laughs.
  DE: Eine Frau mit kurzen blonden Haaren steht von einem Stuhl auf, während eine andere Frau in einem burgunderroten Oberteil lacht.

  EN: A little girl dressed in pink plays hopscotch.
  DE: Ein kleines rosa gekleidetes Mädchen spielt Himmel und Hölle.

  EN: A little girl is playing in the hay.
  DE: Ein kleines Mädchen spielt im Heu.

  EN: Two street artists are performing on steps for a group of spectators.
  DE: Zwei Straßenkünstler geben auf einer Treppe eine Vorstellung für Zuschauergruppe.



## Task 1: Build Vocabularies

Before the model can process text, we need to convert words into numbers. A **vocabulary** is simply a dictionary that maps each unique word to an integer index.

We reserve the first four indices for special tokens that the model needs:

| Index | Token | Purpose |
|-------|-------|---------|
| 0 | `<pad>` | Fill shorter sequences so all items in a batch have the same length |
| 1 | `<unk>` | Represent words not seen during training |
| 2 | `<sos>` | Signal the start of a target sequence (decoder input) |
| 3 | `<eos>` | Signal the end of a sequence |

All regular words get indices starting at 4.

We also need a **reverse mapping** from index back to word so we can decode model outputs into readable text.

First, run the provided tokenizer:

In [ ]:
def simple_tokenize(sentence):
    """Lowercase and split into tokens, separating trailing punctuation."""
    sentence = sentence.lower().strip()
    tokens = []
    for word in sentence.split():
        if word and word[-1] in '.,!?;:':
            tokens.append(word[:-1])
            tokens.append(word[-1])
        else:
            tokens.append(word)
    return [t for t in tokens if t]

# Quick sanity check
print(simple_tokenize("The weather is nice."))
print(simple_tokenize("Ich bin ein Student."))

['the', 'weather', 'is', 'nice', '.']
['ich', 'bin', 'ein', 'student', '.']


Now implement `build_vocab`. The steps are:

1. Tokenize every sentence in the list using `simple_tokenize`
2. Collect all unique tokens into a **sorted** list (sorted so results are reproducible)
3. Build `word2idx` starting with the four special tokens, then append the sorted regular words
4. Build `idx2word` as the reverse of `word2idx`

In [ ]:
from collections import Counter
def build_vocab(sentences):
    """
    Build a vocabulary from a list of sentences.

    Args:
        sentences: list of strings

    Returns:
        word2idx: dict mapping word -> int index
        idx2word: dict mapping int index -> word
    """
    # TODO: Tokenize all sentences and collect unique tokens
    all_tokens = []
    for sentence in sentences:
        sentence_token = simple_tokenize(sentence)
        all_tokens.extend(sentence_token)
    # TODO: Sort unique tokens for reproducibility
    sorted_vocab = []
    counts = Counter(all_tokens)
    sorted_vocab = sorted([token for token, count in counts.items() if count == 1])
    print(f"Number of unique tokens: {len(sorted_vocab)}")
    # TODO: Build word2idx starting with special tokens at indices 0-3
    word2idx = {
        '<pad>': PAD_IDX,
        '<unk>': UNK_IDX,
        '<sos>': SOS_IDX,
        '<eos>': EOS_IDX,
    }
    # Hint: iterate over sorted_vocab and assign indices starting at 4
    for idx, token in enumerate(sorted_vocab, start=4):
      word2idx[token] = idx

    # TODO: Build idx2word as the reverse mapping
    idx2word = {
       idx: word for word, idx in word2idx.items()
    }

    return word2idx, idx2word

In [ ]:
en_sentences = [pair[0] for pair in parallel_data]
de_sentences = [pair[1] for pair in parallel_data]

en_word2idx, en_idx2word = build_vocab(en_sentences)
de_word2idx, de_idx2word = build_vocab(de_sentences)

print(f"English vocabulary size: {len(en_word2idx)}")
print(f"German vocabulary size:  {len(de_word2idx)}")
print(f"\nFirst 10 English vocab entries: {dict(list(en_word2idx.items())[:10])}")
print(f"First 10 German vocab entries:  {dict(list(de_word2idx.items())[:10])}")

Number of unique tokens: 2140
Number of unique tokens: 3645
English vocabulary size: 2144
German vocabulary size:  3649

First 10 English vocab entries: {'<pad>': 0, '<unk>': 1, '<sos>': 2, '<eos>': 3, '"free': 4, '"glass': 5, '"guitar': 6, '"no': 7, '"slow"': 8, '"stop': 9}
First 10 German vocab entries:  {'<pad>': 0, '<unk>': 1, '<sos>': 2, '<eos>': 3, '!': 4, '"blood': 5, '&': 6, '(2': 7, '(einer': 8, '(in': 9}


**Expected output:**

```
English vocabulary size: ~5,000–7,000
German vocabulary size:  ~8,000–12,000

First 10 English vocab entries: {'<pad>': 0, '<unk>': 1, '<sos>': 2, '<eos>': 3, '.': 4, ...}
First 10 German vocab entries:  {'<pad>': 0, '<unk>': 1, '<sos>': 2, '<eos>': 3, '.': 4, ...}
```

Your exact vocabulary sizes will depend on `N_TRAIN`. German vocabulary is much larger than English because German forms compound words that English expresses as multi-word phrases. The first four entries are always the special tokens; regular words start at index 4 in alphabetical order.

## Task 2: Dataset and DataLoader

PyTorch requires data to be wrapped in a `Dataset` object. A `Dataset` must implement two methods:

- `__len__()` — returns the total number of samples
- `__getitem__(idx)` — returns one sample (the source and target sentence as token index lists)

The `DataLoader` handles batching. Because sentences have different lengths, we need a **collate function** that pads all sequences in a batch to the same length. This is provided for you.

We also need a helper `encode_sentence` to convert a raw string into a list of integer indices:

In [ ]:
def encode_sentence(sentence, word2idx):
    """Convert a sentence string to a list of token indices (with <sos> and <eos>)."""
    tokens = ['<sos>'] + simple_tokenize(sentence) + ['<eos>']
    return [word2idx.get(t, UNK_IDX) for t in tokens]

Now implement `TranslationDataset`. Each call to `__getitem__` should return:
- `src`: the encoded English sentence (list of ints)
- `tgt`: the encoded German sentence (list of ints)

In [ ]:
class TranslationDataset(Dataset):
    """
    Dataset for English-to-German translation pairs.

    Each item returns:
        src: list of ints — encoded English sentence with <sos> and <eos>
        tgt: list of ints — encoded German sentence with <sos> and <eos>
    """

    def __init__(self, pairs, en_word2idx, de_word2idx):
        self.pairs = pairs
        self.en_word2idx = en_word2idx
        self.de_word2idx = de_word2idx

    def __len__(self):
        # TODO: Return the number of translation pairs

        return len(self.pairs)

    def __getitem__(self, idx):
        en_sent, de_sent = self.pairs[idx]
        # TODO: Encode the English sentence using encode_sentence
        src = encode_sentence(en_sent, self.en_word2idx)
        # TODO: Encode the German sentence using encode_sentence
        tgt = encode_sentence(de_sent, self.de_word2idx)
        return src, tgt

In [ ]:
def collate_fn(batch):
    """Pad sequences in a batch to the length of the longest sequence."""
    src_batch, tgt_batch = zip(*batch)

    max_src = min(max(len(s) for s in src_batch), MAX_SEQ_LEN)
    max_tgt = min(max(len(t) for t in tgt_batch), MAX_SEQ_LEN)

    padded_src, padded_tgt = [], []
    for src, tgt in zip(src_batch, tgt_batch):
        src = src[:max_src] + [PAD_IDX] * max(0, max_src - len(src))
        tgt = tgt[:max_tgt] + [PAD_IDX] * max(0, max_tgt - len(tgt))
        padded_src.append(src)
        padded_tgt.append(tgt)

    return (
        torch.tensor(padded_src, dtype=torch.long),
        torch.tensor(padded_tgt, dtype=torch.long),
    )


dataset = TranslationDataset(parallel_data, en_word2idx, de_word2idx)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

src_batch, tgt_batch = next(iter(train_loader))
print(f"Source batch shape: {src_batch.shape}")
print(f"Target batch shape: {tgt_batch.shape}")
print(f"First source sequence (indices): {src_batch[0].tolist()}")
print(f"First target sequence (indices): {tgt_batch[0].tolist()}")

Source batch shape: torch.Size([64, 26])
Target batch shape: torch.Size([64, 28])
First source sequence (indices): [2, 1, 1, 1, 1, 1, 1, 1, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
First target sequence (indices): [2, 1, 1, 1, 1, 1, 1, 1, 1, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


**Expected output:**

```
Source batch shape: torch.Size([64, <src_len>])
Target batch shape: torch.Size([64, <tgt_len>])
First source sequence (indices): [2, ..., 3, 0, 0, ...]  # starts with 2 (<sos>), ends with 3 (<eos>), padded with 0
First target sequence (indices): [2, ..., 3, 0, 0, ...]  # same structure
```

Exact lengths depend on the longest sentence in each batch.

## Task 3: Positional Encoding

Transformers process all tokens **simultaneously** (no recurrence), so they have no built-in notion of word order. We fix this by adding a **positional encoding** to each embedding — a vector that encodes the position of a token in the sequence.

We use the sinusoidal positional encoding from the original "Attention Is All You Need" paper:

$$\text{PE}(\text{pos},\, 2i) = \sin\!\left(\frac{\text{pos}}{10000^{\,2i/d_{\text{model}}}}\right)$$

$$\text{PE}(\text{pos},\, 2i+1) = \cos\!\left(\frac{\text{pos}}{10000^{\,2i/d_{\text{model}}}}\right)$$

where `pos` is the position (0, 1, 2, …) and `i` is the dimension index (0, 1, 2, …, d_model/2).

Intuitively:
- Each dimension oscillates at a different frequency
- Low dimensions change rapidly with position; high dimensions change slowly
- Together they create a unique fingerprint for each position

The positional encoding is **pre-computed** once in `__init__` and added to embeddings in `forward`.

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding.

    Adds a position-dependent signal to token embeddings so the model
    can distinguish word order.
    """

    def __init__(self, d_model, max_len=200):
        super().__init__()

        # Pre-compute the positional encoding matrix of shape [max_len, d_model]
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()  # [max_len, 1]

        # TODO: Compute div_term using the formula:
        #   div_term = exp( arange(0, d_model, 2) * -(log(10000.0) / d_model) )
        #   Hint: use torch.arange, torch.exp, torch.log, and torch.tensor
        #   Shape: [d_model // 2]
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(torch.log(torch.tensor(10000.0)) / d_model))

        # TODO: Fill even columns of pe with sin values
        #   pe[:, 0::2] = sin(position * div_term)
        pe[:, 0::2] = torch.sin(position * div_term)

        # TODO: Fill odd columns of pe with cos values
        #   pe[:, 1::2] = cos(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Register pe as a buffer: it is saved with the model but is not a trainable parameter
        self.register_buffer('pe', pe.unsqueeze(0))  # shape: [1, max_len, d_model]

    def forward(self, x):
        """
        Args:
            x: Tensor of shape [batch_size, seq_len, d_model]
        Returns:
            Positional encoding of shape [1, seq_len, d_model] — broadcastable over the batch
        """
        # TODO: Return the positional encoding sliced to match x's sequence length
        #   Hint: self.pe has shape [1, max_len, d_model]; slice the second dimension
        seq_len = x.shape[1]
        return self.pe[:, :seq_len, :]

In [ ]:
pe_module = PositionalEncoding(d_model=D_MODEL)
dummy = torch.zeros(1, 10, D_MODEL)
pe_output = pe_module(dummy)

print(f"PE output shape: {pe_output.shape}")
print(f"\nPE values at position 0 (first 8 dims): {pe_output[0, 0, :8].tolist()}")
print(f"PE values at position 1 (first 8 dims): {pe_output[0, 1, :8].tolist()}")

PE output shape: torch.Size([1, 10, 256])

PE values at position 0 (first 8 dims): [0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0]
PE values at position 1 (first 8 dims): [0.8414709568023682, 0.5403023362159729, 0.8019617795944214, 0.5973753333091736, 0.7617204189300537, 0.6479058265686035, 0.7214140892028809, 0.6925039291381836]


**Expected output:**

```
PE output shape: torch.Size([1, 10, 256])

PE values at position 0 (first 8 dims): [0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0]
PE values at position 1 (first 8 dims): [0.8415, 0.5403, 0.8219, 0.5697, 0.8012, 0.5985, ...]  # varies by d_model
```

At position 0, sin(0) = 0 and cos(0) = 1, so the values alternate 0 and 1 exactly. At other positions, the values are sine/cosine waves.

## Task 4: Seq2Seq Transformer Model

Now we assemble the full encoder-decoder Transformer. Here is a high-level view of the data flow:

```
Source (English indices)
        │
   src_embedding + positional_encoding
        │
   ┌────▼────────┐
   │   Encoder   │  (self-attention over source tokens)
   └────┬────────┘
        │ memory (encoder output)
        │              Target (German indices, shifted right)
        │                     │
        │              tgt_embedding + positional_encoding
        │                     │
   ┌────▼─────────────────────▼──┐
   │          Decoder            │  (self-attention + cross-attention with memory)
   └────────────────┬────────────┘
                    │
              output_projection
                    │
            Logits over German vocabulary
```

We use PyTorch's built-in `nn.Transformer`, which combines encoder and decoder layers internally. This means we do **not** need to write separate Encoder and Decoder classes — `nn.Transformer` handles the cross-attention between them automatically.

### Masks

The Transformer needs three masks:

| Mask | Shape | Purpose |
|------|-------|---------|
| `src_padding_mask` | `[batch, src_len]` | Ignore `<pad>` tokens in the source |
| `tgt_padding_mask` | `[batch, tgt_len]` | Ignore `<pad>` tokens in the target |
| `tgt_causal_mask` | `[tgt_len, tgt_len]` | Prevent the decoder from looking at future tokens |

The `create_mask` helper below builds all three for you.

In [ ]:
def create_mask(src, tgt):
    """
    Build all masks needed by nn.Transformer.

    Args:
        src: [batch_size, src_len] source token indices
        tgt: [batch_size, tgt_len] target token indices

    Returns:
        src_padding_mask: [batch_size, src_len]  — True where src == PAD_IDX
        tgt_padding_mask: [batch_size, tgt_len]  — True where tgt == PAD_IDX
        tgt_causal_mask:  [tgt_len, tgt_len]     — upper-triangular -inf mask
    """
    src_padding_mask = (src == PAD_IDX)          # bool tensor
    tgt_padding_mask = (tgt == PAD_IDX)          # bool tensor
    tgt_len = tgt.shape[1]
    tgt_causal_mask = nn.Transformer.generate_square_subsequent_mask(
        tgt_len, device=tgt.device
    )  # float tensor with -inf above the diagonal
    return src_padding_mask, tgt_padding_mask, tgt_causal_mask

Now implement `Seq2SeqTransformer`.

**`__init__` checklist:**
1. Source embedding (`nn.Embedding` with `padding_idx=PAD_IDX`)
2. Target embedding (`nn.Embedding` with `padding_idx=PAD_IDX`)
3. Positional encoding (your `PositionalEncoding` class)
4. `nn.Transformer` with `batch_first=True`
5. Output linear projection from `d_model` → `tgt_vocab_size`

**`forward` checklist:**
1. Embed source tokens, scale by `sqrt(d_model)`, add positional encoding
2. Embed target tokens, scale by `sqrt(d_model)`, add positional encoding
3. Pass both through `self.transformer` with all masks
4. Project output to vocabulary logits

In [ ]:
class Seq2SeqTransformer(nn.Module):
    """
    Full encoder-decoder Transformer for sequence-to-sequence translation.
    Uses nn.Transformer which handles encoder, decoder, and cross-attention internally.
    """

    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=128,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1,
    ):
        super().__init__()
        self.d_model = d_model

        # TODO: Source token embedding
        #   nn.Embedding(num_embeddings=src_vocab_size, embedding_dim=d_model, padding_idx=PAD_IDX)
        self.src_embedding =  torch.nn.Embedding(num_embeddings=src_vocab_size, embedding_dim=d_model, padding_idx=PAD_IDX)

        # TODO: Target token embedding
        #   nn.Embedding(num_embeddings=tgt_vocab_size, embedding_dim=d_model, padding_idx=PAD_IDX)
        self.tgt_embedding = torch.nn.Embedding(num_embeddings=tgt_vocab_size, embedding_dim=d_model, padding_idx=PAD_IDX)

        # TODO: Positional encoding
        self.pos_encoding = PositionalEncoding(d_model=d_model)

        # TODO: The full Transformer (encoder + decoder)
        #   nn.Transformer(
        #       d_model=d_model,
        #       nhead=nhead,
        #       num_encoder_layers=num_layers,
        #       num_decoder_layers=num_layers,
        #       dim_feedforward=dim_feedforward,
        #       dropout=dropout,
        #       batch_first=True,   # <-- input shape is [batch, seq, d_model]
        #   )
        self.transformer = nn.Transformer(
               d_model=d_model,
               nhead=nhead,
               num_encoder_layers=num_layers,
               num_decoder_layers=num_layers,
               dim_feedforward=dim_feedforward,
               dropout=dropout,
               batch_first=True,   # <-- input shape is [batch, seq, d_model]
           )

        # TODO: Linear layer that maps d_model -> tgt_vocab_size (produces logits)
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)

        self.dropout = nn.Dropout(dropout)

    def forward(self, src, tgt):
        """
        Args:
            src: [batch_size, src_len] source token indices
            tgt: [batch_size, tgt_len] target token indices

        Returns:
            logits: [batch_size, tgt_len, tgt_vocab_size]
        """
        src_padding_mask, tgt_padding_mask, tgt_causal_mask = create_mask(src, tgt)

        # TODO: Embed source, scale by sqrt(d_model), add positional encoding, apply dropout
        #   src_emb = self.dropout(self.src_embedding(src) * math.sqrt(self.d_model) + self.pos_encoding(...))
        src_embedded = self.src_embedding(src)
        src_emb = self.dropout(src_embedded * math.sqrt(self.d_model) + self.pos_encoding(src_embedded))

        # TODO: Embed target, scale by sqrt(d_model), add positional encoding, apply dropout
        tgt_embedded = self.tgt_embedding(tgt)
        tgt_emb = self.dropout(tgt_embedded * math.sqrt(self.d_model) + self.pos_encoding(tgt_embedded))

        # TODO: Pass through the Transformer
        #   self.transformer(
        #       src_emb, tgt_emb,
        #       tgt_mask=tgt_causal_mask,
        #       src_key_padding_mask=src_padding_mask,
        #       tgt_key_padding_mask=tgt_padding_mask,
        #       memory_key_padding_mask=src_padding_mask,
        #   )
        output = self.transformer(
               src_emb, tgt_emb,
               tgt_mask=tgt_causal_mask,
               src_key_padding_mask=src_padding_mask,
               tgt_key_padding_mask=tgt_padding_mask,
               memory_key_padding_mask=src_padding_mask,
        )
        # TODO: Project to vocabulary logits
        logits = self.output_projection(output)

        return logits

In [ ]:
SRC_VOCAB_SIZE = len(en_word2idx)
TGT_VOCAB_SIZE = len(de_word2idx)

model = Seq2SeqTransformer(
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {total_params:,}")

# Quick forward pass to verify shapes
src_batch, tgt_batch = next(iter(train_loader))
src_batch, tgt_batch = src_batch.to(device), tgt_batch.to(device)
tgt_input = tgt_batch[:, :-1]   # decoder input: all tokens except the last

with torch.no_grad():
    out = model(src_batch, tgt_input)

print(f"\nForward pass test:")
print(f"  src shape:    {src_batch.shape}")
print(f"  tgt_in shape: {tgt_input.shape}")
print(f"  output shape: {out.shape}   <-- should be [batch, tgt_len-1, tgt_vocab_size]")

Total trainable parameters: 5,057,601


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(



Forward pass test:
  src shape:    torch.Size([64, 28])
  tgt_in shape: torch.Size([64, 29])
  output shape: torch.Size([64, 29, 3649])   <-- should be [batch, tgt_len-1, tgt_vocab_size]


**Expected output:**

```
Total trainable parameters: ~7,000,000

Forward pass test:
  src shape:    torch.Size([64, <src_len>])
  tgt_in shape: torch.Size([64, <tgt_len - 1>])
  output shape: torch.Size([64, <tgt_len - 1>, <de_vocab_size>])   <-- should be [batch, tgt_len-1, tgt_vocab_size]
```

The output's last dimension is the German vocabulary size. The model produces a probability distribution over all German words for each target position. Parameter count is higher than on the toy corpus because the embedding layers scale with vocabulary size.

## Task 5: Training Loop

### Teacher Forcing

During training we use **teacher forcing**: we feed the model the *correct* previous token as decoder input, rather than its own (potentially wrong) prediction. This speeds up training and keeps gradients stable.

For a target sequence `[<sos>, w1, w2, w3, <eos>]`:
- **Decoder input** (`tgt_input`): `[<sos>, w1, w2, w3]` — all tokens *except* the last
- **Labels** (`tgt_labels`):       `[w1, w2, w3, <eos>]` — all tokens *except* the first

At each position the model sees the correct prefix and must predict the next token.

### Loss

We use `CrossEntropyLoss` with `ignore_index=PAD_IDX` so that padding positions do not contribute to the loss.

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion):
    """
    Train the model for one full pass over the dataset.

    Args:
        model:      Seq2SeqTransformer
        dataloader: training DataLoader
        optimizer:  Adam optimizer
        criterion:  CrossEntropyLoss

    Returns:
        avg_loss: average loss over all batches in this epoch
    """
    model.train()
    total_loss = 0

    for src, tgt in dataloader:
        src, tgt = src.to(device), tgt.to(device)

        # TODO: Create decoder input (tgt without its last token)
        tgt_input = tgt[:, :-1]

        # TODO: Create labels (tgt without its first token)
        tgt_labels = tgt[:, 1:]

        # TODO: Zero the optimizer gradients
        optimizer.zero_grad()

        # TODO: Forward pass — get logits from the model
        logits = model(src, tgt_input)

        # Reshape for CrossEntropyLoss:
        #   logits:     [batch * tgt_len, vocab_size]
        #   tgt_labels: [batch * tgt_len]
        loss = criterion(
            logits.reshape(-1, logits.shape[-1]),
            tgt_labels.reshape(-1),
        )

        # TODO: Backward pass
        loss.backward()

        # TODO: Update model weights
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

print("Training...")
for epoch in range(NUM_EPOCHS):
    loss = train_one_epoch(model, train_loader, optimizer, criterion)
    if (epoch + 1) % 3 == 0:
        print(f"  Epoch {epoch + 1:2d}/{NUM_EPOCHS}  |  Loss: {loss:.4f}")
print("Done.")

Training...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


  Epoch  3/15  |  Loss: 0.8610
  Epoch  6/15  |  Loss: 0.8172
  Epoch  9/15  |  Loss: 0.7946
  Epoch 12/15  |  Loss: 0.7647
  Epoch 15/15  |  Loss: 0.7391
Done.


**Expected output:**

```
Training...
  Epoch  3/15  |  Loss: ~5.5
  Epoch  6/15  |  Loss: ~4.0
  Epoch  9/15  |  Loss: ~3.2
  Epoch 12/15  |  Loss: ~2.7
  Epoch 15/15  |  Loss: ~2.3
Done.
```

The loss should decrease steadily over epochs. With a real dataset the model will not memorise all training pairs perfectly — final loss is higher than on a tiny toy corpus — but the translations will generalise far better to new sentences. Exact values depend on `N_TRAIN` and random initialisation.

## Task 6: Greedy Translation

After training, we want to use the model to translate new sentences. During inference we **cannot** use teacher forcing because we don't have the correct German tokens — the model must generate them one at a time.

We use **greedy decoding**: at each step, pick the token with the highest probability and feed it back as the next decoder input.

The algorithm:
1. Encode the source sentence → `src_tensor`
2. Start the decoder input with `[<sos>]`
3. **Repeat:**
   - Run the model with current source and decoder input
   - Take the logits at the **last** position
   - Pick `argmax` → predicted token
   - Append to decoder input
   - **Stop** if predicted token is `<eos>` or max length is reached
4. Convert token indices to words (skip `<sos>` and `<eos>`)

In [ ]:
def translate(model, sentence, en_word2idx, de_idx2word, max_len=MAX_SEQ_LEN):
    """
    Translate an English sentence into German using greedy decoding.

    Args:
        model:        trained Seq2SeqTransformer
        sentence:     English string
        en_word2idx:  English word-to-index mapping
        de_idx2word:  German index-to-word mapping
        max_len:      maximum number of generated tokens

    Returns:
        translated:   German string
    """
    model.eval()

    src_indices = encode_sentence(sentence, en_word2idx)
    src_tensor = torch.tensor([src_indices], dtype=torch.long).to(device)

    tgt_indices = [SOS_IDX]  # start decoder with <sos>

    with torch.no_grad():
        for _ in range(max_len):
            tgt_tensor = torch.tensor([tgt_indices], dtype=torch.long).to(device)

            # TODO: Run the model to get logits
            #   Shape: [1, current_tgt_len, tgt_vocab_size]
            logits = model(src_tensor, tgt_tensor)

            # TODO: Get the predicted token — argmax over vocabulary at the LAST position
            #   Hint: logits[0, -1, :] gives the logits for the last position
            next_token = logits[0, -1, :].argmax(dim=-1).item()

            # TODO: Append the predicted token index (as a Python int) to tgt_indices
            tgt_indices.append(next_token)

            # TODO: Break if the predicted token is EOS_IDX
            if next_token == EOS_IDX:
                break

    # Decode indices to words (skip <sos>, stop before <eos>)
    words = []
    for idx in tgt_indices[1:]:
        if idx == EOS_IDX:
            break
        words.append(de_idx2word.get(idx, '<unk>'))

    return ' '.join(words)

In [ ]:
test_sentences = [
    "A man is walking down the street.",
    "Two dogs are playing in the park.",
    "A woman is reading a book.",
    "The children are running outside.",
    "A group of people are sitting together.",
]

print("Translation Results")
print("=" * 60)
for en_sent in test_sentences:
    de_sent = translate(model, en_sent, en_word2idx, de_idx2word)
    print(f"  EN: {en_sent}")
    print(f"  DE: {de_sent}")
    print("-" * 60)

Translation Results
  EN: A man is walking down the street.
  DE: <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk>
------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


  EN: Two dogs are playing in the park.
  DE: <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk>
------------------------------------------------------------
  EN: A woman is reading a book.
  DE: <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk>
------------------------------------------------------------
  EN: The children are running outside.
  DE: <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk>
------------------------------------------------------------
  EN: A group of people are sitting together.
  DE: <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk>
------------------------------------------------------------


**Expected output:**

```
Translation Results
============================================================
  EN: A man is walking down the street.
  DE: ein mann geht die straße entlang .
------------------------------------------------------------
  EN: Two dogs are playing in the park.
  DE: zwei hunde spielen im park .
------------------------------------------------------------
  EN: A woman is reading a book.
  DE: eine frau liest ein buch .
------------------------------------------------------------
  EN: The children are running outside.
  DE: die kinder laufen draußen .
------------------------------------------------------------
  EN: A group of people are sitting together.
  DE: eine gruppe von menschen sitzt zusammen .
------------------------------------------------------------
```

Exact translations will vary with random initialisation and `N_TRAIN`, but they should be grammatically plausible and capture the meaning of each sentence. Unlike the toy corpus, the model now generalises to sentences it has not seen verbatim during training.

## Conclusion

Congratulations! You have built a complete encoder-decoder Transformer for English-to-German translation from scratch, and trained it on a real-world dataset.

Here is a summary of what each task covered:

| Task | What you built | Key concept |
|------|---------------|-------------|
| 1 | `build_vocab` | Word-to-index mapping with special tokens |
| 2 | `TranslationDataset` | PyTorch Dataset and DataLoader for parallel text |
| 3 | `PositionalEncoding` | Sinusoidal position signals |
| 4 | `Seq2SeqTransformer` | Full encoder-decoder model with `nn.Transformer` |
| 5 | `train_one_epoch` | Teacher forcing and cross-entropy training |
| 6 | `translate` | Autoregressive greedy decoding |

### Limitations and Next Steps

Our model is trained on Multi30k (~29k image-caption pairs). Production-quality translation systems go further:

- **Much larger datasets** — WMT datasets contain millions of sentence pairs; modern systems train on hundreds of billions of tokens
- **Subword tokenization** — BPE or SentencePiece instead of word-level splitting; handles unseen words gracefully and reduces vocabulary size significantly
- **Beam search** — instead of greedy decoding, keep the top-k most promising sequences at each step for higher-quality translations
- **BLEU score** — the standard automatic metric for evaluating translation quality
- **Pre-trained models** — Helsinki-NLP MarianMT and similar models are already trained on large corpora and can be fine-tuned for specific domains